In [1]:
from openai import AsyncOpenAI


In [7]:
client = AsyncOpenAI(
    base_url="https://api-aipro.chatbaram.com/sllm/v1",
    api_key="EMPTY"
)

In [8]:
json_schema = {
    "type": "object",
    "properties": {
        "city": {"type": "string"},
        "country": {"type": "string"},
        "population": {"type": "integer"},
        "landmarks": {
            "type": "array",
            "items": {"type": "string"}
        }
    },
    "required": ["city", "country", "population", "landmarks"]
}

In [9]:
response = await client.chat.completions.create(
    model="google/gemma-4-26B-A4B-it",
    messages=[
        {
            "role": "system",
            "content": "Extract city information as structured JSON."
        },
        {
            "role": "user",
            "content": "Tell me about Paris, France."
        }
    ],
    response_format={
        "type": "json_schema",
        "json_schema": {
            "name": "city-info",
            "schema": json_schema
        }
    },
    max_tokens=512
)

In [10]:
import json
data = json.loads(response.choices[0].message.content)
print(data)

{'city': 'Paris', 'country': 'France', 'population': 8, 'landmarks': ['Eiffel Tower', 'Louvre Museum', 'Notre-Dame Cathedral', 'Arc de Triomphe']}


In [12]:
from typing import Optional
from pydantic import BaseModel, Field
from openai import OpenAI

client = OpenAI(
    base_url="https://api-aipro.chatbaram.com/sllm/v1",
    api_key="EMPTY"
)

class WeatherReport(BaseModel):
    air_temperature: Optional[float] = Field(None, description="Temperature in Fahrenheit")
    wind_speed: Optional[float] = Field(None, description="Wind speed in mph")
    comments_or_answer: str = Field(description="Comments or answer to the user's question")

response = client.chat.completions.create(
    model="google/gemma-4-26B-A4B-it",
    messages=[
        {
            "role": "system",
            "content": (
                "Extract the weather information. Output JSON with these fields:\n"
                "- air_temperature: float, converted to Fahrenheit\n"
                "- wind_speed: float, converted to mph\n"
                "- comments_or_answer: string, answer the user's question"
            )
        },
        {
            "role": "user",
            "content": "The current weather in Seattle is 22.0°C with a wind speed of 6.0 km/h."
        }
    ],
    response_format={
        "type": "json_schema",
        "json_schema": {
            "name": "weather-report",
            "schema": WeatherReport.model_json_schema()
        }
    },
    max_tokens=256
)

In [14]:
print(response.choices[0].message.content)

{"air_temperature": 71.6, "wind_speed": 3.73, "comments_or_answer": "The current weather in Seattle is 22.0°C with a wind speed of 6.0 km/h."}


In [15]:
response = client.chat.completions.create(
    model="google/gemma-4-26B-A4B-it",
    messages=[
        {
            "role": "system",
            "content": (
                "Analyze the text and extract entities. Output JSON with:\n"
                "- people: list of person names mentioned\n"
                "- organizations: list of organization names\n"
                "- locations: list of location names\n"
                "- summary: one-sentence summary of the text"
            )
        },
        {
            "role": "user",
            "content": "Dr. Elena Torres, lead researcher at the Riverside Institute, presented her findings on marine biodiversity at the annual symposium in Cape Marina. The Oceanic Wildlife Fund and the Global Conservation Alliance both pledged support."
        }
    ],
    response_format={
        "type": "json_schema",
        "json_schema": {
            "name": "entity-extraction",
            "schema": {
                "type": "object",
                "properties": {
                    "people": {"type": "array", "items": {"type": "string"}},
                    "organizations": {"type": "array", "items": {"type": "string"}},
                    "locations": {"type": "array", "items": {"type": "string"}},
                    "summary": {"type": "string"}
                },
                "required": ["people", "organizations", "locations", "summary"]
            }
        }
    },
    max_tokens=4096,
    extra_body={
        "chat_template_kwargs": {"enable_thinking": True}
    }
)

message = response.choices[0].message

if hasattr(message, "reasoning") and message.reasoning:
    print("=== Thinking ===")
    print(message.reasoning)

print("\n=== Structured Output ===")
print(message.content)


=== Structured Output ===
{"people": ["Dr. Elena Torres"], "organizations": ["Riverside Institute", "Oceanic Wildlife Fund", "Global Conservation Alliance"], "locations": ["Cape Marina"], "summary": "Dr. Elena Torres of the Riverside Institute presented her research on marine biodiversity at a symposium in Cape Marina, receiving support from the Oceanic Wildlife Fund and the Global Conservation Alliance."}
